# Carnage-Lite (1.2M) — Colab training

This notebook builds the C++ trainer on a Colab GPU, resumes the newest Drive checkpoint, and runs one session for up to about 5.4 hours. It syncs completed checkpoints and CSV metrics to Drive and makes one dated checkpoint backup per day.

Before running:
1. Set the runtime to a T4 GPU.
2. This notebook clones the public Carnage-Lite-Training repository at https://github.com/vfxjamer/Carnage-Lite-Training.
3. Check that Google Drive has enough space for the recent checkpoints, 32 ELO policy versions, and daily backups.

The model starts from fresh weights. This export trains only Phases 1 and 2 in 1v1, with phase selection kept manual after reviewing gameplay and metrics. Training targets 1B total timesteps and stops at the first iteration boundary at or beyond that target. Phases 3–5 and the planned 2v2 segment are paused for a later update. Tick skip is 8.

The trainer writes mode-specific self-play ELO to checkpoints/elo_history.csv and per-reward episode metrics split by Blue and Orange to checkpoints/reward_episode_metrics.csv. It has no W&B dependency or replay restore.


In [1]:
from pathlib import Path
import time

# Edit these settings before running the training cell.
REPO_URL = "https://github.com/vfxjamer/Carnage-Lite-Training"  # Public project repository.
PHASE = 1      # Choose 1 or 2 manually after reviewing the checks in README.md.
POLICY_LR_OVERRIDE = None  # Optional float, e.g. 0.00008; None uses the phase default.
GAMES = 2048
SESSION_HOURS = 5.4
SYNC_SECONDS = 60
SESSION_START_MONOTONIC = time.monotonic()

ROOT = Path("/content/Carnage-Lite")
BUILD = ROOT / "build"
LOCAL_CHECKPOINTS = BUILD / "checkpoints"
DRIVE_ROOT = Path("/content/drive/MyDrive/Carnage-Lite")
DRIVE_CHECKPOINTS = DRIVE_ROOT / "checkpoints"

if not REPO_URL.startswith("https://github.com/"):
    raise ValueError("Set REPO_URL to the public GitHub HTTPS clone URL, then rerun this cell.")
if PHASE not in (1, 2):
    raise ValueError("PHASE must be 1 or 2.")
if GAMES < 1:
    raise ValueError("GAMES must be positive.")

print("Configuration ready.")


Configuration ready.


In [2]:
import subprocess

if (ROOT / ".git").is_dir():
    subprocess.run(["git", "-C", str(ROOT), "pull", "--ff-only"], check=True)
elif ROOT.exists() and any(ROOT.iterdir()):
    raise RuntimeError(f"{ROOT} exists but is not a Git checkout. Move it aside and rerun.")
else:
    subprocess.run(["git", "clone", REPO_URL, str(ROOT)], check=True)

print("Project:", ROOT)
print("Commit:", subprocess.check_output(["git", "-C", str(ROOT), "rev-parse", "--short", "HEAD"], text=True).strip())


Project: /content/Carnage-Lite
Commit: 2c06699


In [3]:
import shutil
from google.colab import drive

drive.mount("/content/drive")
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
DRIVE_CHECKPOINTS.mkdir(parents=True, exist_ok=True)
LOCAL_CHECKPOINTS.mkdir(parents=True, exist_ok=True)

if any(DRIVE_CHECKPOINTS.iterdir()):
    shutil.copytree(DRIVE_CHECKPOINTS, LOCAL_CHECKPOINTS, dirs_exist_ok=True)

def numbered_dirs(parent):
    if not parent.exists():
        return []
    found = []
    for item in parent.iterdir():
        if item.is_dir() and item.name.isdigit():
            found.append(item)
    return sorted(found, key=lambda item: int(item.name))

restored = numbered_dirs(LOCAL_CHECKPOINTS)
print("Restored training checkpoints:", [p.name for p in restored[-8:]])
print("Restored policy versions:", len(numbered_dirs(LOCAL_CHECKPOINTS / "policy_versions")))


Mounted at /content/drive
Restored training checkpoints: []
Restored policy versions: 0


In [4]:
import os
import subprocess
import torch

subprocess.run(["apt-get", "update", "-qq"], check=True)
subprocess.run(
    ["apt-get", "install", "-y", "-qq", "build-essential", "cmake", "git", "libpython3-dev", "pkg-config"],
    check=True,
)

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("Select a GPU runtime in Colab, then rerun this cell.")
print("GPU:", torch.cuda.get_device_name(0))
subprocess.run(["nvidia-smi"], check=True)

subprocess.run(
    ["cmake", "-S", str(ROOT), "-B", str(BUILD), "-DCMAKE_BUILD_TYPE=Release"],
    check=True,
)
subprocess.run(
    ["cmake", "--build", str(BUILD), "--config", "Release", "-j", str(os.cpu_count() or 2)],
    check=True,
)

CANDIDATES = [BUILD / "Carnage", BUILD / "Carnage.exe"]
TRAINER = next((p for p in CANDIDATES if p.is_file()), None)
if TRAINER is None:
    raise FileNotFoundError(f"Build finished but no Carnage executable was found under {BUILD}")
print("Trainer:", TRAINER)


PyTorch: 2.14.1+cu130
CUDA available: True
GPU: Tesla T4
Trainer: /content/Carnage-Lite/build/Carnage


## Training session

The SIGTERM at the session boundary is handled by the learner at an iteration boundary; it saves before exiting. The supervisor also restarts nonzero exits with backoff. Completed checkpoint folders are copied through temporary folders and renamed on Drive so a sync does not expose a half-copied checkpoint.

The current phase and optional LR override are set in the first code cell. GigaLearnCPP reports SB3 Clip Fraction in its iteration metrics; its PPO clip range is 0.2 by default, and it has no automatic clip-fraction controller.


In [ ]:
import datetime
import json
import signal
import threading
import time
import shutil
import subprocess

SESSION_SECONDS = int(SESSION_HOURS * 60 * 60)
LOG_FILE = BUILD / "train.log"
SYNC_STATE = {}
SYNCED_STATE = {}
SYNC_STOP = threading.Event()

def tree_signature(folder):
    try:
        rows = []
        for item in sorted(folder.rglob("*")):
            if item.is_file():
                stat = item.stat()
                rows.append((str(item.relative_to(folder)), stat.st_size, stat.st_mtime_ns))
        return tuple(rows)
    except (FileNotFoundError, PermissionError, OSError):
        return None

def is_complete_snapshot(folder, stats_name, min_model_files=2):
    if not (folder / stats_name).is_file():
        return False
    model_files = [p for p in folder.glob("*.lt") if p.is_file() and p.stat().st_size > 0]
    return len(model_files) >= min_model_files

def atomic_copy_tree(source, destination, expected_signature):
    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary = destination.with_name(destination.name + ".syncing")
    if temporary.exists():
        shutil.rmtree(temporary)
    try:
        shutil.copytree(source, temporary)
        if tree_signature(source) != expected_signature:
            shutil.rmtree(temporary, ignore_errors=True)
            return False
        if destination.exists():
            shutil.rmtree(destination)
        temporary.replace(destination)
        return True
    except OSError as exc:
        shutil.rmtree(temporary, ignore_errors=True)
        print("Checkpoint copy will retry:", exc)
        return False

def copy_when_stable(source, destination, key, stats_name, force=False, min_model_files=2):
    if not is_complete_snapshot(source, stats_name, min_model_files):
        return False
    signature = tree_signature(source)
    if signature is None:
        return False
    if destination.exists() and tree_signature(destination) == signature:
        SYNC_STATE[key] = signature
        SYNCED_STATE[key] = signature
        return True
    previous = SYNC_STATE.get(key)
    if not force and signature != previous:
        SYNC_STATE[key] = signature
        return False
    if not force and SYNCED_STATE.get(key) == signature and destination.exists():
        return True
    copied = atomic_copy_tree(source, destination, signature)
    if copied:
        SYNC_STATE[key] = signature
        SYNCED_STATE[key] = signature
    return copied

def copy_csv(source, destination):
    if not source.is_file():
        return
    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary = destination.with_name(destination.name + ".syncing")
    shutil.copy2(source, temporary)
    temporary.replace(destination)

def prune_remote(remote_parent, keep_names):
    if not remote_parent.exists():
        return
    for item in remote_parent.iterdir():
        if item.is_dir() and item.name.isdigit() and item.name not in keep_names:
            shutil.rmtree(item, ignore_errors=True)

def sync_once(force=False):
    local_versions = LOCAL_CHECKPOINTS / "policy_versions"
    remote_versions = DRIVE_CHECKPOINTS / "policy_versions"

    ready_checkpoints = []
    for source in numbered_dirs(LOCAL_CHECKPOINTS):
        if not is_complete_snapshot(source, "RUNNING_STATS.json"):
            continue
        ready_checkpoints.append(source)
        copy_when_stable(
            source,
            DRIVE_CHECKPOINTS / source.name,
            "checkpoint:" + source.name,
            "RUNNING_STATS.json",
            force=force,
        )

    ready_versions = []
    for source in numbered_dirs(local_versions):
        if not is_complete_snapshot(source, "STATS.json", min_model_files=1):
            continue
        ready_versions.append(source)
        copy_when_stable(
            source,
            remote_versions / source.name,
            "policy:" + source.name,
            "STATS.json",
            force=force,
            min_model_files=1,
        )

    if ready_checkpoints:
        prune_remote(DRIVE_CHECKPOINTS, {p.name for p in ready_checkpoints})
    if ready_versions:
        prune_remote(remote_versions, {p.name for p in ready_versions})

    for name in ("elo_history.csv", "reward_episode_metrics.csv", "mode_state.json.bak", "mode_state.json"):
        copy_csv(LOCAL_CHECKPOINTS / name, DRIVE_CHECKPOINTS / name)

def maybe_daily_backup():
    ready = [
        p for p in numbered_dirs(DRIVE_CHECKPOINTS)
        if is_complete_snapshot(p, "RUNNING_STATS.json")
    ]
    if not ready:
        return
    latest = ready[-1]
    day = datetime.datetime.now(datetime.timezone.utc).strftime("%Y-%m-%d")
    backup_root = DRIVE_ROOT / "daily_backups"
    backup = backup_root / day
    if backup.exists():
        return
    temporary = backup_root / (day + ".syncing")
    shutil.rmtree(temporary, ignore_errors=True)
    temporary.mkdir(parents=True, exist_ok=True)
    shutil.copytree(latest, temporary / "checkpoint")
    for name in ("elo_history.csv", "reward_episode_metrics.csv", "mode_state.json.bak", "mode_state.json"):
        source = DRIVE_CHECKPOINTS / name
        if source.is_file():
            shutil.copy2(source, temporary / name)
    metadata = {
        "created_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(),
        "checkpoint_timesteps": int(latest.name),
        "source": str(DRIVE_CHECKPOINTS),
    }
    (temporary / "backup.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")
    backup_root.mkdir(parents=True, exist_ok=True)
    temporary.replace(backup)
    print("Created daily backup:", backup)

def sync_worker():
    while not SYNC_STOP.wait(SYNC_SECONDS):
        try:
            sync_once()
            maybe_daily_backup()
        except Exception as exc:
            print("Drive sync will retry:", repr(exc))

SYNC_THREAD = threading.Thread(target=sync_worker, name="drive-sync", daemon=True)
SYNC_THREAD.start()

command = [
    str(TRAINER),
    str(ROOT / "collision_meshes"),
    "--device", "cuda",
    "--mode", "auto",
    "--phase", str(PHASE),
    "--games", str(GAMES),
    "--save-dir", str(LOCAL_CHECKPOINTS),
]
if POLICY_LR_OVERRIDE is not None:
    command.extend(["--policy-lr", str(POLICY_LR_OVERRIDE)])

print("Starting:", " ".join(command), flush=True)
deadline = SESSION_START_MONOTONIC + SESSION_SECONDS
active_process = None
restart_delay = 10

def drain_output(process):
    with LOG_FILE.open("a", encoding="utf-8", errors="replace") as log:
        for line in iter(process.stdout.readline, ""):
            print(line, end="", flush=True)
            log.write(line)
            log.flush()

try:
    while time.monotonic() < deadline:
        remaining = deadline - time.monotonic()
        if remaining <= 120:
            break
        active_process = subprocess.Popen(
            command,
            cwd=ROOT,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        reader = threading.Thread(target=drain_output, args=(active_process,), daemon=True)
        reader.start()

        try:
            code = active_process.wait(timeout=remaining - 90)
        except subprocess.TimeoutExpired:
            print("Session window reached; requesting a safe iteration-boundary save.", flush=True)
            active_process.send_signal(signal.SIGTERM)
            try:
                code = active_process.wait(timeout=120)
            except subprocess.TimeoutExpired:
                print("Trainer did not stop after SIGTERM; forcing process exit.", flush=True)
                active_process.kill()
                code = active_process.wait()
            reader.join(timeout=10)
            break

        reader.join(timeout=10)
        if code == 0:
            print("Trainer reached its configured timestep target or exited cleanly.", flush=True)
            break

        print(f"Trainer exited with code {code}; retrying after {restart_delay}s.", flush=True)
        if time.monotonic() + restart_delay + 120 >= deadline:
            break
        time.sleep(restart_delay)
        restart_delay = min(restart_delay * 2, 120)
finally:
    if active_process is not None and active_process.poll() is None:
        active_process.send_signal(signal.SIGTERM)
        try:
            active_process.wait(timeout=120)
        except subprocess.TimeoutExpired:
            active_process.kill()
            active_process.wait()
    SYNC_STOP.set()
    SYNC_THREAD.join(timeout=10)
    sync_once(force=True)
    maybe_daily_backup()

print("Session finished. Checkpoints and CSV metrics are under:", DRIVE_CHECKPOINTS)


Starting: /content/Carnage-Lite/build/Carnage /content/Carnage-Lite/collision_meshes --device cuda --mode auto --phase 1 --games 2048 --save-dir /content/Carnage-Lite/build/checkpoints
Carnage-Lite (8.9M)
Mode: 1v1 | phase: 1 | checkpoint step: 0 | initial team spirit: 0
Initializing RocketSim version 2.1.1, created by ZealanL...
Loading arena meshes for soccar...
   > Loaded 18 verts and 16 tris, hash: 0x1f8ee550
   > Loaded 536 verts and 983 tris, hash: 0x94fb0d5c
   > Loaded 80 verts and 126 tris, hash: 0x760358d3
   > Loaded 80 verts and 126 tris, hash: 0xb81ac8b9
   > Loaded 483 verts and 880 tris, hash: 0x2811eee8
   > Loaded 483 verts and 880 tris, hash: 0xec759ebf
   > Loaded 18 verts and 16 tris, hash: 0x255ba8c1
   > Loaded 18 verts and 16 tris, hash: 0xd84c7a68
   > Loaded 18 verts and 16 tris, hash: 0x3d79d25d
   > Loaded 483 verts and 880 tris, hash: 0xa160baf9
   > Loaded 80 verts and 126 tris, hash: 0x73ae4940
   > Loaded 483 verts and 880 tris, hash: 0x14b84668
   > Loa

In [ ]:
from pathlib import Path

def print_tail(path, count=12):
    path = Path(path)
    print("\n" + str(path))
    if not path.exists():
        print("not created yet")
        return
    for line in path.read_text(encoding="utf-8", errors="replace").splitlines()[-count:]:
        print(line)

print_tail(DRIVE_CHECKPOINTS / "elo_history.csv")
print_tail(DRIVE_CHECKPOINTS / "reward_episode_metrics.csv")
print("Latest Drive checkpoint:", numbered_dirs(DRIVE_CHECKPOINTS)[-1].name if numbered_dirs(DRIVE_CHECKPOINTS) else "none")
